# 21장 · 문서를 정리하고 원문 위치를 보존하며 청킹하기

도서관 안내에서 ‘14일입니다.’를 찾았는데 잘라 둔 조각이 ‘일입니다.’로 시작한다면 숫자는 어디로 갔을까요? 글을 자른 위치와 앞 조각에 겹친 부분을 원문과 나란히 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 문자 경계와 겹치는 부분 찾기

청크는 검색에 넣으려고 나눈 원문의 한 조각입니다. 이 실험에서는 파이썬 문자열의 문자 수를 기준으로 16자씩 자르고, 이웃 조각 사이에 4자를 겹칩니다. 모델 토큰 수로 자른 실험과는 단위가 다릅니다.



**볼 곳**　금색 ‘반납 기한은’이 원문에 남은 채 오른쪽 조각에도 생기는 순간을 보세요.

**애니메이션의 의미**　다음 조각에도 앞말을 남겨 ‘14일’의 뜻을 보존합니다.


### 셀 01 · 첫 16자 읽기

20장 첫 문서의 온점이 첫 조각에 들어가는지 보세요.

In [1]:
doc = {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1}
text = doc["text"]
first = text[:16]
print(first)

도서 대출 기간은 14일입니다


‘도서 대출 기간은 14일입니다’까지 들어가고 맨 끝 온점은 다음에 남습니다. 범위의 끝 번호 16은 포함하지 않습니다.

### 셀 02 · 둘째 조각의 시작점

시작점을 16에서 겹칠 4만큼 앞으로 당깁니다.

첫 조각은 text[0:16]으로 인덱스 0~15를 담습니다. 끝점 16은 포함하지 않습니다. 네 글자를 겹치면 둘째 조각은 16−4=12에서 시작합니다. 인덱스 16은 처음부터 세어 열일곱 번째 글자입니다.

In [2]:
second = text[16 - 4:]
print(second)

일입니다.


둘째 조각에는 ‘일입니다.’가 들어 있습니다. 서로 이어 붙일 때 이 네 글자가 두 번 나타날 수 있습니다. 입력 문장을 바꾸지 않고 시작 위치만 확인하면 이유를 볼 수 있습니다.

### 과제 03 · 겹치는 네 글자 확인하기

첫 조각 끝 4자와 둘째 조각 앞 4자가 같은지 검사하고 그 문자열을 출력하세요.

In [3]:
# 첫 조각의 끝과 둘째 조각의 앞을 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>assert first[-4:] == second[:4]
print(first[-4:])</code></pre>
<p>실행 결과</p><pre><code>일입니다</code></pre>
<p>‘일입니다’가 두 조각에 같이 들어 있습니다. 이 결과는 온점 직전까지의 겹침을 설명합니다.</p>
</details>

## 조각에서 원문의 위치 찾아가기

검색 결과에 조각 본문만 담으면 근거가 어느 자료 몇 쪽인지 잊기 쉽습니다. 표시할 문서에는 id, text, source, page를 두고 원문의 위치는 별도 표에 남길 수 있습니다.



**볼 곳**　‘위치 정보’를 기준으로 양쪽의 ‘library-1:1’와 ‘원문 1쪽’를 대조하세요.

**애니메이션의 의미**　두 번째 조각에서 원문 식별자와 쪽을 거슬러 올라갑니다.


### 셀 04 · 둘째 조각의 문서와 위치

실험용 식별자에는 0부터 센 조각 번호를 붙입니다.

In [4]:
chunk = {"id": f"{doc['id']}:1", "text": second, "source": doc["source"], "page": doc["page"]}
location = {"document_id": doc["id"], "start": 12, "end": len(text)}
print(chunk["id"], text[location["start"]:location["end"]])

library-1:1 일입니다.


library-1:1과 ‘일입니다.’가 나옵니다. ID만 읽어 위치를 추측하지 않고 기록한 범위에서 본문을 되찾았습니다.

### 과제 05 · 겹침 없이 다시 자르기

크기는 16으로 유지하고 중첩만 0으로 바꾸세요. 둘째 조각에 무엇이 남나요?

In [5]:
# 같은 원문의 16번째 위치부터 잘라 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(text[16:])</code></pre>
<p>실행 결과</p><pre><code>.</code></pre>
<p>온점 한 글자가 남습니다. 조각을 더 많이 만들었다고 검색에 쓸 근거가 항상 더 좋아지는 것은 아닙니다.</p>
</details>

## 실험 결과와 다음 장의 입력 구분하기

다음 장에서는 세 자료에 각각 하나의 벡터를 연결하려고 공통 기준 자료를 씁니다. 이 기준에서는 한 쪽 전체가 한 조각이며 ID는 library-1 그대로입니다. 방금 만든 library-1:1은 크기 비교용으로만 보관합니다.



**볼 곳**　‘같은 원문’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘분할 실험’, 아래쪽은 ‘기준 자료’입니다.

**애니메이션의 의미**　분할 실험을 마쳐도 원본 문서는 따로 보존합니다.


### 셀 06 · 다음 장에 넘길 ID 확인하기

기준 문서 한 건에서 출처가 있는 본문을 통째로 읽습니다.

In [6]:
print(doc["id"], doc["text"])

library-1 도서 대출 기간은 14일입니다.


library-1의 전체 대출 문장이 온점까지 남습니다. 청킹 방식을 바꾼 실험을 정식 입력으로 채택하려면 원문과 청크를 대응하는 표와 이후 벡터도 함께 새로 만들어야 합니다.

## 돌아보기

원문을 합칠 때는 겹친 구간을 한 번만 세고, 출처를 표시할 때는 원문 ID와 조각 ID 중 어떤 단위를 쓰는지 알려 주세요. 눈으로 차이를 구분한 뒤 size와 overlap을 한 번에 하나만 바꿔 보세요.